# 10. Финальная оценка mDeBERTa на test

Использована исходная модель `text_standard`, seed 42, из сохранённого Output Kaggle `notebook80586b8724`. Дополнительный seed 2026 не заменяет её. Для исходного выполнения требовались GPU, CSV и `bank_risk_multiclass_dl/runs/text_standard/best_model/model.safetensors`.

Код не содержит повторного обучения. Флаг `RUN_FINAL_TEST=False` по умолчанию блокирует оценку. Результаты выполненного запуска сохранены отдельно в локальном проекте.

In [4]:
import os,sys,subprocess,importlib.util
os.environ['USE_TF']='0';os.environ['USE_FLAX']='0';os.environ['TOKENIZERS_PARALLELISM']='false'
missing=[p for p in ['transformers','sentencepiece','safetensors'] if importlib.util.find_spec(p) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
# torch._utils нужен до ленивой загрузки DeBERTa.
import importlib
import torch
torch_utils = importlib.import_module('torch._utils')
if not hasattr(torch, '_utils'):
    torch._utils = torch_utils
assert hasattr(torch._utils, '_get_device_index'), 'torch._utils неполон; требуется перезапуск сессии Kaggle.'
importlib.import_module('torch._dynamo')
print('PyTorch:', torch.__version__)
import numpy as np, pandas as pd, json, hashlib
from pathlib import Path
from transformers import AutoTokenizer,AutoModelForSequenceClassification
from sklearn.metrics import f1_score,precision_score,recall_score,accuracy_score,confusion_matrix
CLASSES=['no_risk','cyber_risk','operational_risk','regulatory_risk']
DATA_SHA='52333a13e7fb6f6c4162486198fbf1d1ab368672a8974295995580a1924f67f3'
inputs=Path('/kaggle/input')
assert inputs.is_dir(),'Этот ноутбук предназначен для Kaggle с подключёнными Input.'
cs=[p for p in inputs.rglob('bank_risk_dataset.csv') if hashlib.sha256(p.read_bytes()).hexdigest()==DATA_SHA]
assert cs,'В Kaggle Input отсутствует исходный bank_risk_dataset.csv.'
source=cs[0]; data=pd.read_csv(source)
assert len(data)==5041 and data['sample_id'].is_unique
assert set(data['risk_type_4cls'])==set(CLASSES)
model_dirs=sorted({p.parent for p in inputs.rglob('model.safetensors')
                   if p.parent.name=='best_model' and p.parent.parent.name=='text_standard'
                   and 'bank_risk_multiclass_dl' in p.parts})
assert len(model_dirs)==1, ('В Kaggle Input требуется сохранённый Output исходного обучения '
   'notebook80586b8724 с единственной моделью text_standard/seed42. Найдено:',model_dirs)
MODEL_DIR=model_dirs[0]
configpath=MODEL_DIR.parent/'run_config.json'
if configpath.is_file():
    cfg=json.loads(configpath.read_text('utf8'))
    assert cfg['config']['seed']==42 and cfg['config']['mode']=='text' and cfg['config']['loss_mode']=='standard'
    assert cfg['dataset_sha256']==DATA_SHA
print('Датасет:', source)
print('Замороженная модель:',MODEL_DIR)
print('Test примеров:',len(data.loc[data['split']=='test']))
assert len(data.loc[data['split']=='test'])==751

PyTorch: 2.11.0+cu128
Датасет: /kaggle/input/datasets/victordoynikov/bank-risks/bank_risk_dataset.csv
Замороженная модель: /kaggle/input/notebooks/victordoynikov/notebook80586b8724/bank_risk_multiclass_dl/runs/text_standard/best_model
Test примеров: 751


In [5]:
RUN_FINAL_TEST = False
OUT = Path('/kaggle/working/bank_risk_multiclass_final_test')
metrics_path = OUT/'test_metrics.json'
if not RUN_FINAL_TEST:
    print('Test отключён: RUN_FINAL_TEST=False.')
elif metrics_path.exists():
    print('Сохранённый test:',metrics_path)
    print(json.loads(metrics_path.read_text('utf8')))
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()
    model_labels = model.config.id2label
    assert all(model_labels[i] == CLASSES[i] for i in range(4)),model_labels
    test=data.loc[data['split']=='test',['sample_id','risk_type_4cls','text_fragment']].copy().reset_index(drop=True)
    texts=test['text_fragment'].fillna('').astype(str).tolist()
    logits=[]
    with torch.inference_mode():
        for i in range(0,len(texts),8):
            inputs=tokenizer(texts[i:i+8],padding=True,truncation=True,max_length=256,return_tensors='pt')
            outputs=model(**{k:v.to(device) for k,v in inputs.items()})
            logits.append(outputs.logits.detach().float().cpu().numpy())
    scores=np.concatenate(logits,axis=0)
    assert scores.shape==(751,4)
    pred_ids=scores.argmax(axis=1)
    pred_labels=np.array(CLASSES)[pred_ids]
    truth=test['risk_type_4cls'].to_numpy()
    class_f1=f1_score(truth,pred_labels,labels=CLASSES,average=None,zero_division=0)
    metrics={'model':'text_standard','family':'deep_learning','seed':42,'split':'test','dataset_sha256':DATA_SHA,
        'n_samples':len(test),'macro_f1':float(class_f1.mean()),'risk_macro_f1':float(class_f1[1:].mean()),
        'macro_precision':float(precision_score(truth,pred_labels,labels=CLASSES,average='macro',zero_division=0)),
        'macro_recall':float(recall_score(truth,pred_labels,labels=CLASSES,average='macro',zero_division=0)),
        'weighted_f1':float(f1_score(truth,pred_labels,labels=CLASSES,average='weighted',zero_division=0)),
        'accuracy':float(accuracy_score(truth,pred_labels)),
        'confusion_matrix':confusion_matrix(truth,pred_labels,labels=CLASSES).tolist()}
    metrics.update({'f1_'+c:float(s) for c,s in zip(CLASSES,class_f1)})
    OUT.mkdir(parents=True,exist_ok=True)
    metrics_path.write_text(json.dumps(metrics,ensure_ascii=False,indent=2),'utf8')
    dfout=test[['sample_id','risk_type_4cls']].rename(columns={'risk_type_4cls':'true'}).copy()
    dfout['prediction']=pred_labels
    for j,c in enumerate(CLASSES): dfout['logit_'+c]=scores[:,j]
    dfout.to_csv(OUT/'test_predictions.csv',index=False)
    print('Финальный test завершён. Macro F1:',metrics['macro_f1'])
    print('Результаты:',OUT)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Финальный test завершён. Macro F1: 0.7329368906955114
Результаты: /kaggle/working/bank_risk_multiclass_final_test


## Сохранённый результат

Метрики и предсказания находятся в `results/multiclass/final_test/dl/`. Test Macro F1 равен 0,7329. Ноутбук 11 объединяет их с результатами baseline; дальнейший подбор по test не выполняется. Output относится к завершённой оценке; повторный запуск отключён (`RUN_FINAL_TEST=False`).